# Lab 22-11 · Machines: a DC Motor Start, the Induction Equivalent Circuit and Torque–Speed Curves

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate
import machines as mc

VPH = 400 / math.sqrt(3)


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Starting a DC motor

*Predict* the final speed and current, the mechanical time constant J Ra/k^2, and the current peak.

In [2]:
def f(t, y):
    i, w = y
    return [(24 - 0.5 * i - 0.05 * w) / 1e-3, (0.05 * i - 0.5 * min(1.0, max(0.0, w))) / 1e-4]
sol = integrate.solve_ivp(f, (0, 0.2), [0, 0], max_step=1e-5, rtol=1e-8, atol=1e-10)
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(sol.t * 1e3, sol.y[0]); ax[0].set_ylabel("current (A)")
ax[1].plot(sol.t * 1e3, sol.y[1]); ax[1].set_ylabel("speed (rad/s)"); ax[1].set_xlabel("time (ms)"); plt.show()
print(f"peak {sol.y[0].max():.1f} A, final {sol.y[0][-1]:.2f} A, {sol.y[1][-1]:.1f} rad/s; tau_m = {1e-4*0.5/0.05**2*1e3:.0f} ms")
s = ngspice("dcmotor.cir")
print({kk: s[kk] for kk in ("wfin", "ifin", "ipk", "t63")})

peak 41.7 A, final 10.00 A, 380.0 rad/s; tau_m = 20 ms
{'wfin': 379.9943, 'ifin': 10.00064, 'ipk': 41.65165, 't63': 0.02048299}


## Part 2 · The equivalent circuit in ngspice

*Predict* Example 22-11.4's starting and running torque and current.

In [3]:
m = mc.Induction(VPH, 50, 4, 0.5, 1.0, 30.0, 0.4, 1.0)
for sl in (1.0, 0.03):
    print(f"s = {sl}: T = {m.torque(sl):.1f} N m, I1 = {abs(m.stator_current(sl)):.1f} A")
s = ngspice("induction.cir")
print({kk: s[kk] for kk in ("tst", "ist", "trun", "irun")})

s = 1.0: T = 81.9 N m, I1 = 107.0 A
s = 0.03: T = 65.4 N m, I1 = 18.0 A
{'tst': 81.94165, 'ist': 107.0282, 'trun': 65.41659, 'irun': 18.02902}


## Part 3 · Rotor resistance

*Predict* how R2 = 0.4, 1.0 and 2.03 ohm change the curve at standstill, at the peak and near synchronous speed.

In [4]:
sl = np.linspace(1e-4, 1, 1000)
plt.figure()
for r2 in (0.4, 1.0, 2.03):
    plt.plot((1 - sl) * 1500, [m.torque(x, r2) for x in sl], label=f"R2 = {r2}")
    print(f"R2 = {r2}: s_max = {m.s_max(r2):.3f}, T(1) = {m.torque(1.0, r2):.1f} N m")
plt.xlabel("rpm"); plt.ylabel("N m"); plt.legend(); plt.show()

R2 = 0.4: s_max = 0.197, T(1) = 81.9 N m
R2 = 1.0: s_max = 0.493, T(1) = 157.4 N m
R2 = 2.03: s_max = 1.000, T(1) = 190.9 N m


## Part 4 · Constant V/f

*Predict* how the curves move from 50 Hz down to 20 Hz, and the peak torque at 20 Hz.

In [5]:
plt.figure()
for f in (50, 40, 30, 20):
    k = f / 50
    mf = mc.Induction(VPH * k, f, 4, 0.5, 1.0 * k, 30.0 * k, 0.4, 1.0 * k)
    plt.plot((1 - sl) * mc.sync_speed_rpm(f, 4), [mf.torque(x) for x in sl], label=f"{f} Hz")
    print(f"{f} Hz: peak torque {mf.t_max():.1f} N m")
plt.xlabel("rpm"); plt.ylabel("N m"); plt.legend(); plt.show()

50 Hz: peak torque 190.9 N m
40 Hz: peak torque 180.0 N m
30 Hz: peak torque 163.4 N m
20 Hz: peak torque 136.1 N m


## Part 5 · The EV envelope

*Predict* the power at base speed and the torque at 15 000 rpm.

In [6]:
n = np.linspace(1, 15000, 500)
t = np.array([mc.ev_envelope(400, 5000, x) for x in n])
plt.figure(); plt.plot(n, t, label="torque (N m)"); plt.plot(n, t * n * mc.RPM / 1e3, label="power (kW)")
plt.xlabel("rpm"); plt.legend(); plt.show()
print(f"base power {400*5000*mc.RPM/1e3:.0f} kW; torque at 15 000 rpm {mc.ev_envelope(400, 5000, 15000):.0f} N m")

base power 209 kW; torque at 15 000 rpm 133 N m


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.